# Latent Diffusion para ArtBench-10

Este notebook implementa uma versão **starter** de **Latent Diffusion** para o projeto ArtBench.

Ideia:
1. treinar primeiro um **Autoencoder convolucional** para comprimir imagens 32×32 para um espaço latente;
2. treinar um **modelo de difusão no espaço latente** em vez do espaço de pixels;
3. gerar latentes por difusão e depois reconstruir imagens com o decoder.

Este notebook foi pensado para:
- usar o **starter pack do professor**;
- carregar o **subset de 20%** com `training_20_percent.csv`;
- treinar primeiro no subset;
- guardar os pesos do autoencoder e do diffusion model.

> Isto é uma implementação simplificada, adequada para o projeto e para comparação com VAE / GAN / DDPM.

## 1. Imports

In [ ]:
from __future__ import annotations

import sys
import csv
import math
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.utils import make_grid

## 2. Configuração

In [ ]:
SEED = 42
IMAGE_SIZE = 32
BATCH_SIZE = 64
NUM_WORKERS = 0

# Autoencoder
AE_EPOCHS = 20
AE_LR = 1e-3
LATENT_CHANNELS = 4
LATENT_SIZE = 8   # 32x32 -> 8x8 latent map

# Diffusion no espaço latente
DIFF_EPOCHS = 20
DIFF_LR = 2e-4
TIMESTEPS = 200
BETA_START = 1e-4
BETA_END = 0.02

SAVE_EVERY = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 3. Caminhos

In [ ]:
PROJECT_ROOT = Path("..")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
KAGGLE_ROOT = PROJECT_ROOT.parent / "ArtBench-10"
TRAINING_CSV_PATH = PROJECT_ROOT / "training_20_percent.csv"

print("PROJECT_ROOT =", PROJECT_ROOT.resolve())
print("SCRIPTS_DIR  =", SCRIPTS_DIR.resolve())
print("KAGGLE_ROOT  =", KAGGLE_ROOT.resolve())
print("CSV          =", TRAINING_CSV_PATH.resolve())

assert SCRIPTS_DIR.exists(), f"Não existe: {SCRIPTS_DIR.resolve()}"
assert KAGGLE_ROOT.exists(), f"Não existe: {KAGGLE_ROOT.resolve()}"
assert TRAINING_CSV_PATH.exists(), f"Não existe: {TRAINING_CSV_PATH.resolve()}"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

## 4. Dataset com o helper do professor

In [ ]:
from artbench_local_dataset import load_kaggle_artbench10_splits

hf_ds = load_kaggle_artbench10_splits(KAGGLE_ROOT)
train_hf = hf_ds["train"]

label_feature = train_hf.features["label"]
class_names = list(label_feature.names)

print("Train size:", len(train_hf))
print("Número de classes:", len(class_names))
print("Classes:", class_names)

## 5. Transform e dataset PyTorch

In [ ]:
transform = T.Compose([
    T.Resize(IMAGE_SIZE, interpolation=T.InterpolationMode.BILINEAR),
    T.CenterCrop(IMAGE_SIZE),
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

class HFDatasetTorch(Dataset):
    def __init__(self, hf_split, transform=None, indices=None):
        self.ds = hf_split
        self.transform = transform
        self.indices = list(range(len(hf_split))) if indices is None else list(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        ex = self.ds[real_idx]
        img = ex["image"]
        y = int(ex["label"])
        x = self.transform(img) if self.transform is not None else img
        return x, y, real_idx

def denorm(x):
    return (x * 0.5 + 0.5).clamp(0, 1)

## 6. Carregar o subset de 20%

In [ ]:
INDEX_COLUMN = "train_id_original"

def load_ids_from_training_csv(csv_path: Path, index_column: str = "train_id_original") -> list[int]:
    ids = []
    with open(csv_path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        if index_column not in (reader.fieldnames or []):
            raise ValueError(f"Coluna {index_column!r} não encontrada. Disponíveis: {reader.fieldnames}")
        for row in reader:
            value = str(row.get(index_column, "")).strip()
            if value:
                ids.append(int(value))
    if len(ids) == 0:
        raise ValueError("Não foram lidos IDs do CSV.")
    return ids

train_ids_from_csv = load_ids_from_training_csv(TRAINING_CSV_PATH, INDEX_COLUMN)

train_ds = HFDatasetTorch(train_hf, transform=transform, indices=train_ids_from_csv)
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("Subset size:", len(train_ds))
print("Batches:", len(train_loader))

## 7. Ver imagens reais

In [ ]:
x, y, idx = next(iter(train_loader))
grid = make_grid(denorm(x[:36]), nrow=6, padding=2)

plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
plt.axis("off")
plt.title("Imagens reais do subset")
plt.show()

## 8. Autoencoder convolucional

O latent diffusion precisa primeiro de um encoder/decoder.
Aqui usamos um autoencoder simples:
- Encoder: 32×32 -> 8×8 latente
- Decoder: 8×8 latente -> 32×32

In [ ]:
class ConvAutoencoder(nn.Module):
    def __init__(self, latent_channels=4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1),   # 32 -> 16
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1),  # 16 -> 8
            nn.ReLU(),
            nn.Conv2d(64, latent_channels, 3, 1, 1)  # 8 -> 8
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(latent_channels, 64, 3, 1, 1),  # 8 -> 8
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),               # 8 -> 16
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1),                # 16 -> 32
            nn.Tanh()
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z

## 9. Treinar o Autoencoder

In [ ]:
autoencoder = ConvAutoencoder(latent_channels=LATENT_CHANNELS).to(device)
ae_optimizer = torch.optim.Adam(autoencoder.parameters(), lr=AE_LR)

def train_autoencoder(model, dataloader, optimizer, epochs, device):
    losses = []

    print("Training Autoencoder...")
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for x, _, _ in dataloader:
            x = x.to(device)
            x_hat, z = model(x)

            recon_loss = F.mse_loss(x_hat, x)

            optimizer.zero_grad()
            recon_loss.backward()
            optimizer.step()

            running_loss += recon_loss.item()

        avg_loss = running_loss / len(dataloader)
        losses.append(avg_loss)
        print(f"[AE] Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.6f}")

    return losses

ae_losses = train_autoencoder(autoencoder, train_loader, ae_optimizer, AE_EPOCHS, device)

## 10. Curva de loss do Autoencoder

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(ae_losses)
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Autoencoder reconstruction loss")
plt.grid(True, alpha=0.3)
plt.show()

## 11. Ver reconstruções do Autoencoder

In [ ]:
autoencoder.eval()
with torch.no_grad():
    x_batch, _, _ = next(iter(train_loader))
    x_batch = x_batch.to(device)[:16]
    x_hat, z = autoencoder(x_batch)

real_grid = make_grid(denorm(x_batch.cpu()), nrow=4, padding=2)
recon_grid = make_grid(denorm(x_hat.cpu()), nrow=4, padding=2)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(real_grid.permute(1, 2, 0))
plt.axis("off")
plt.title("Reais")

plt.subplot(1, 2, 2)
plt.imshow(recon_grid.permute(1, 2, 0))
plt.axis("off")
plt.title("Reconstruções")
plt.tight_layout()
plt.show()

print("Latent shape:", z.shape)

## 12. Embeddings de tempo para o diffusion no espaço latente

In [ ]:
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        device = t.device
        half_dim = self.dim // 2
        emb_factor = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb_factor)
        emb = t[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=1)
        return emb

## 13. U-Net pequena para latentes

Agora o modelo já não trabalha sobre imagens 3×32×32, mas sim sobre latentes 4×8×8.

In [ ]:
class LatentBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim):
        super().__init__()
        self.time_mlp = nn.Linear(time_emb_dim, out_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.act = nn.SiLU()

    def forward(self, x, t_emb):
        h = self.act(self.conv1(x))
        h = h + self.time_mlp(t_emb).unsqueeze(-1).unsqueeze(-1)
        h = self.act(self.conv2(h))
        return h


class LatentUNet(nn.Module):
    def __init__(self, latent_channels=4, base_ch=64, time_emb_dim=128):
        super().__init__()
        self.time_emb = nn.Sequential(
            SinusoidalPosEmb(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim),
        )

        self.down1 = LatentBlock(latent_channels, base_ch, time_emb_dim)
        self.pool1 = nn.MaxPool2d(2)  # 8 -> 4

        self.mid = LatentBlock(base_ch, base_ch * 2, time_emb_dim)

        self.up = nn.ConvTranspose2d(base_ch * 2, base_ch, 2, stride=2)  # 4 -> 8
        self.up_block = LatentBlock(base_ch * 2, base_ch, time_emb_dim)

        self.out = nn.Conv2d(base_ch, latent_channels, kernel_size=1)

    def forward(self, x, t):
        t_emb = self.time_emb(t)

        h1 = self.down1(x, t_emb)
        p1 = self.pool1(h1)

        mid = self.mid(p1, t_emb)

        up = self.up(mid)
        cat = torch.cat([up, h1], dim=1)
        h = self.up_block(cat, t_emb)

        return self.out(h)

## 14. Processo de difusão nos latentes

In [ ]:
betas = torch.linspace(BETA_START, BETA_END, TIMESTEPS).to(device)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
alphas_cumprod_prev = torch.cat([torch.tensor([1.0], device=device), alphas_cumprod[:-1]], dim=0)

sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)
sqrt_recip_alphas = torch.sqrt(1.0 / alphas)

posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)

In [ ]:
def q_sample(x_start, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x_start)

    sqrt_alpha_t = sqrt_alphas_cumprod[t].view(-1, 1, 1, 1)
    sqrt_one_minus_alpha_t = sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1)

    return sqrt_alpha_t * x_start + sqrt_one_minus_alpha_t * noise, noise

## 15. Inicializar o diffusion model

In [ ]:
latent_model = LatentUNet(latent_channels=LATENT_CHANNELS, base_ch=64, time_emb_dim=128).to(device)
latent_optimizer = torch.optim.Adam(latent_model.parameters(), lr=DIFF_LR)

print("Latent diffusion params:", sum(p.numel() for p in latent_model.parameters()))

## 16. Treinar diffusion no espaço latente

Aqui congelamos o autoencoder e treinamos apenas o diffusion model sobre os latentes.

In [ ]:
def train_latent_diffusion(autoencoder, latent_model, dataloader, optimizer, epochs, device):
    losses = []

    autoencoder.eval()
    print("Training Latent Diffusion...")

    for epoch in range(epochs):
        latent_model.train()
        running_loss = 0.0

        for x, _, _ in dataloader:
            x = x.to(device)

            with torch.no_grad():
                z = autoencoder.encode(x)

            bsz = z.size(0)
            t = torch.randint(0, TIMESTEPS, (bsz,), device=device).long()

            noisy_z, noise = q_sample(z, t)
            pred_noise = latent_model(noisy_z, t)

            loss = F.mse_loss(pred_noise, noise)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(dataloader)
        losses.append(avg_loss)
        print(f"[LDM] Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.6f}")

    return losses

ldm_losses = train_latent_diffusion(autoencoder, latent_model, train_loader, latent_optimizer, DIFF_EPOCHS, device)

## 17. Curva de loss do Latent Diffusion

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(ldm_losses)
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Latent diffusion loss")
plt.grid(True, alpha=0.3)
plt.show()

## 18. Sampling reverso no espaço latente

In [ ]:
@torch.no_grad()
def p_sample(model, x, t):
    betas_t = betas[t].view(-1, 1, 1, 1)
    sqrt_one_minus_alphas_cumprod_t = sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1)
    sqrt_recip_alphas_t = sqrt_recip_alphas[t].view(-1, 1, 1, 1)

    model_mean = sqrt_recip_alphas_t * (
        x - betas_t * model(x, t) / sqrt_one_minus_alphas_cumprod_t
    )

    posterior_variance_t = posterior_variance[t].view(-1, 1, 1, 1)

    noise = torch.randn_like(x)
    nonzero_mask = (t != 0).float().view(-1, 1, 1, 1)

    return model_mean + nonzero_mask * torch.sqrt(posterior_variance_t) * noise


@torch.no_grad()
def sample_latent_diffusion(autoencoder, latent_model, n=16):
    autoencoder.eval()
    latent_model.eval()

    z = torch.randn(n, LATENT_CHANNELS, LATENT_SIZE, LATENT_SIZE, device=device)

    for i in reversed(range(TIMESTEPS)):
        t = torch.full((n,), i, device=device, dtype=torch.long)
        z = p_sample(latent_model, z, t)

    x_hat = autoencoder.decode(z)
    return denorm(x_hat.cpu())

## 19. Gerar imagens

In [ ]:
samples = sample_latent_diffusion(autoencoder, latent_model, n=64)

grid = make_grid(samples, nrow=8, padding=2)
plt.figure(figsize=(10, 10))
plt.imshow(grid.permute(1, 2, 0))
plt.axis("off")
plt.title("Amostras geradas por Latent Diffusion")
plt.show()

## 20. Guardar os modelos

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "checkpoints"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

torch.save(autoencoder.state_dict(), OUTPUT_DIR / "latent_diffusion_autoencoder_subset20.pth")
torch.save(latent_model.state_dict(), OUTPUT_DIR / "latent_diffusion_model_subset20.pth")

print("Modelos guardados em:", OUTPUT_DIR.resolve())

## 21. Próximos passos

1. testar mais épocas no autoencoder e no diffusion model;
2. experimentar `TIMESTEPS = 500`;
3. usar um autoencoder mais forte;
4. criar notebook de avaliação para gerar 5000 imagens e calcular FID/KID;
5. comparar com o DDPM em pixels, VAE e GAN.

Este notebook deve ser visto como uma **base funcional** para latent diffusion no projeto.